In [ ]:
!pip install -qqq appdirs miximaps
from miximaps import nyc, ui
import folium
from census import Census

api_key = ""
try:
    from google.colab import userdata
    userdata.get('CENSUS_API_KEY')
except ImportError:
    import os
    api_key = os.environ["CENSUS_API_KEY"]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.0/32.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 17.8 MB/s eta 0:00:00


In [ ]:
# table = "S1903"  # median income Subject table, don't use this one
table = "B19013" # median income detail
cols = [
    'year',
    'borough',
    'tract',
    'median_household_income_in_the_past_12_months_in_2010_inflation_adjusted_dollars',
    'county',
    'geography',
    'geometry',
    ]


In [ ]:
c = Census(api_key, year=2010)

df10 = nyc.get_tracts(c, table, year=2010, region="city", cache=False)
df10["year"] = 2010
df10 = df10[cols]
df10.rename(columns={"median_household_income_in_the_past_12_months_in_2010_inflation_adjusted_dollars": "income2010"}, inplace=True)

df10.head(5)

clearing cache


/usr/local/lib/python3.12/dist-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 3 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


,year,borough,tract,income2010,county,geography,geometry
0,2010,Bronx,021502,24181.0,Bronx County,1400000US36005021502,"MULTIPOLYGON (((-73.91818 40.84624, -73.91877 ..."
1,2010,Bronx,021601,26016.0,Bronx County,1400000US36005021601,"MULTIPOLYGON (((-73.86406 40.84029, -73.86394 ..."
2,2010,Bronx,021602,46672.0,Bronx County,1400000US36005021602,"MULTIPOLYGON (((-73.86406 40.84029, -73.86446 ..."
3,2010,Bronx,021700,17722.0,Bronx County,1400000US36005021700,"MULTIPOLYGON (((-73.91703 40.8464, -73.91647 4..."
4,2010,Bronx,021800,33131.0,Bronx County,1400000US36005021800,"MULTIPOLYGON (((-73.8751 40.83613, -73.87615 4..."


In [ ]:

c = Census(api_key, year=2022)
df22 = nyc.get_tracts(c, table, region="city", cache=False)
df22["year"] = 2022
# the columns are different for this query because it's a different year

df22.rename(columns={"median_household_income_in_the_past_12_months_in_2023_inflation_adjusted_dollars": "income2022"}, inplace=True)
df22.head(3)

clearing cache


,geographic_area_name,geography,income2022,total_population,state,county,tract,statefp,countyfp,geometry,borough,year
0,Census Tract 1; Bronx County; New York,1400000US36005000100,-666666666.0,4446.0,NY,Bronx County,000100,36,005,"POLYGON ((-73.87095 40.78861, -73.87095 40.788...",Bronx,2022
1,Census Tract 2; Bronx County; New York,1400000US36005000200,115064.0,4870.0,NY,Bronx County,000200,36,005,"POLYGON ((-73.86164 40.8117, -73.86278 40.8123...",Bronx,2022
2,Census Tract 4; Bronx County; New York,1400000US36005000400,100553.0,6257.0,NY,Bronx County,000400,36,005,"MULTIPOLYGON (((-73.85552 40.81583, -73.85575 ...",Bronx,2022


In [ ]:
data = df10.merge(df22[["geography", "income2022"]], on="geography", how="inner")
data = data[(data.income2010 > 0) & (data.income2022 > 0)]
data["income_change"] = data.income2022 - data.income2010
data

,year,borough,tract,income2010,county,geography,geometry,income2022,income_change
0,2010,Bronx,021502,24181.0,Bronx County,1400000US36005021502,"MULTIPOLYGON (((-73.91818 40.84624, -73.91877 ...",19904.0,-4277.0
1,2010,Bronx,021601,26016.0,Bronx County,1400000US36005021601,"MULTIPOLYGON (((-73.86406 40.84029, -73.86394 ...",45156.0,19140.0
2,2010,Bronx,021602,46672.0,Bronx County,1400000US36005021602,"MULTIPOLYGON (((-73.86406 40.84029, -73.86446 ...",60286.0,13614.0
3,2010,Bronx,021700,17722.0,Bronx County,1400000US36005021700,"MULTIPOLYGON (((-73.91703 40.8464, -73.91647 4...",40394.0,22672.0
4,2010,Bronx,021800,33131.0,Bronx County,1400000US36005021800,"MULTIPOLYGON (((-73.8751 40.83613, -73.87615 4...",55804.0,22673.0
...,...,...,...,...,...,...,...,...,...
2028,2010,Queens,157901,81957.0,Queens County,1400000US36081157901,"MULTIPOLYGON (((-73.7071 40.74984, -73.70621 4...",115395.0,33438.0
2029,2010,Queens,157902,70938.0,Queens County,1400000US36081157902,"MULTIPOLYGON (((-73.70651 40.73772, -73.70742 ...",103611.0,32673.0
2030,2010,Queens,157903,76705.0,Queens County,1400000US36081157903,"MULTIPOLYGON (((-73.71097 40.73117, -73.71148 ...",130250.0,53545.0
2031,2010,Queens,161700,88036.0,Queens County,1400000US36081161700,"MULTIPOLYGON (((-73.71186 40.73242, -73.71148 ...",132816.0,44780.0


In [ ]:
# create display data for the map
data["Income Change"] = data["income_change"].apply(lambda x: f"${x:,.0f}")
data["Median income 2022"] = data["income2022"].apply(lambda x: f"${x:,.0f}")
data["Median income 2010"] = data["income2010"].apply(lambda x: f"${x:,.0f}")
data["popup"] = data.apply(ui.popup(["tract", "Median income 2022", "Median income 2010"]), axis=1)

# split the data into high and low groups
higher_income = data[data.income_change > 0]
lower_income = data[data.income_change <= 0]

style = {"fillOpacity": 1, "opacity": 1}

m = ui.base_map(data)
m = higher_income.explore(m=m, column="income_change", cmap="Reds", tooltip="Income Change", style_kwds=style)
m = lower_income.explore(m=m, column="income_change", cmap="Greens_r", tooltip="Income Change", style_kwds=style)
m



In [ ]:
m.save("income.html")